# Lakehouse On-Premise — Demo en vivo

**Metricraft Workshop · Hora 2**

Este notebook muestra el flujo completo sobre Iceberg:

1. Ingesta del CSV → tabla `demo.sales.orders`
2. Segunda carga (genera un snapshot extra)
3. Historial / **time travel**
4. **Schema evolution** + update

> Ejecutá cada celda con **Shift+Enter**. No corras todo de un jalón si querés narrar la demo.

## 0) Sesión Spark

En la arquitectura de **2 VPS**, el catálogo y MinIO están en vps01 (`10.116.0.3`),
no en los hosts Docker `rest` / `minio`.

**Reglas:**
1. Si ves `Another SparkContext` → **Kernel → Restart Kernel and Clear All Outputs**
2. Ejecutá **solo una vez** la celda de abajo
3. Los `.config(...)` apuntan a la IP privada del VPS1


In [ ]:
from pyspark.sql import SparkSession

# IP privada VPC de vps01 (storage + catálogo). NO usar rest:8181 ni minio:9000.
VPS1 = "10.116.0.3"

# Si ya hay sesión (a menudo con defaults rest/minio), cerrarla para aplicar configs nuevas.
active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder.appName("lakehouse-demo-vivo")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.demo.uri", f"http://{VPS1}:8181")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.demo.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.demo.s3.endpoint", f"http://{VPS1}:9000")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.catalog.demo.s3.access-key-id", "admin")
    .config("spark.sql.catalog.demo.s3.secret-access-key", "password12345")
    .config("spark.sql.catalog.demo.client.region", "us-east-1")
    .config("spark.sql.defaultCatalog", "demo")
    .config("spark.hadoop.fs.s3a.endpoint", f"http://{VPS1}:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password12345")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Catalog URI:", spark.conf.get("spark.sql.catalog.demo.uri"))
print("S3 endpoint:", spark.conf.get("spark.sql.catalog.demo.s3.endpoint"))
spark


## 1) Ingesta — leer el CSV de ejemplo

Fuente: `/home/iceberg/data/sample_sales.csv` (10 filas de ventas ficticias).

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.sales")

df = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .csv("/home/iceberg/data/sample_sales.csv")
)

print(f"Filas leídas: {df.count()}")
df.show(truncate=False)

## 2) Escribir como tabla Iceberg

Los datos van a MinIO (Parquet) y los metadatos al catálogo REST + Postgres.

In [ ]:
df.writeTo("demo.sales.orders").createOrReplace()

print("Tabla creada: demo.sales.orders")
spark.sql("SELECT * FROM demo.sales.orders").show(truncate=False)

## 3) Segunda carga (nuevo snapshot)

Sin un segundo snapshot no se puede demostrar time travel en vivo.

In [ ]:
from datetime import date
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# order_date debe ser DATE (igual que el CSV con inferSchema), no STRING
extra_schema = StructType(
    [
        StructField("order_id", IntegerType(), False),
        StructField("customer", StringType(), False),
        StructField("product", StringType(), False),
        StructField("amount", DoubleType(), False),
        StructField("order_date", DateType(), False),
    ]
)
extra = spark.createDataFrame(
    [(1011, "Empresa Andina SAC", "Licencia ERP", 1200.50, date(2026, 1, 25))],
    extra_schema,
)
extra.printSchema()
extra.writeTo("demo.sales.orders").append()

print("Segunda carga aplicada")
spark.sql("SELECT * FROM demo.sales.orders ORDER BY order_id").show(truncate=False)

## 4) Historial de snapshots

Iceberg guarda cada commit. Anotá el **primer** `snapshot_id` para el time travel.

In [ ]:
print("=== history ===")
spark.sql("SELECT * FROM demo.sales.orders.history").show(truncate=False)

print("=== snapshots ===")
spark.sql(
    "SELECT snapshot_id, committed_at, operation FROM demo.sales.orders.snapshots"
).show(truncate=False)

snapshots = spark.sql(
    "SELECT snapshot_id FROM demo.sales.orders.snapshots ORDER BY committed_at"
).collect()
first_snapshot = snapshots[0]["snapshot_id"]
print(f"Primer snapshot_id (usar en time travel): {first_snapshot}")

## 5) Time travel

Consultamos la tabla **como estaba antes** de la segunda carga (sin la orden 1011).

In [ ]:
print(f"VERSION AS OF {first_snapshot}")
spark.sql(
    f"SELECT * FROM demo.sales.orders VERSION AS OF {first_snapshot} ORDER BY order_id"
).show(truncate=False)

print("Estado actual (incluye orden 1011):")
spark.sql("SELECT * FROM demo.sales.orders ORDER BY order_id").show(truncate=False)

## 6) Schema evolution

Agregamos la columna `region` **sin reescribir** los archivos Parquet existentes. Las filas viejas quedan en `NULL`.

> Si re-ejecutás esta celda y la columna ya existe, Spark devolverá error: es esperado. Seguí con la celda 7 o reiniciá la tabla desde la celda 2.

In [ ]:
spark.sql("ALTER TABLE demo.sales.orders ADD COLUMN region STRING")

spark.sql("SELECT * FROM demo.sales.orders ORDER BY order_id").show(truncate=False)

## 7) Update con el nuevo campo

Iceberg soporta `UPDATE` (ACID). Queda registrado como otro snapshot.

In [ ]:
spark.sql(
    "UPDATE demo.sales.orders SET region = 'Lima' WHERE customer = 'Empresa Andina SAC'"
)

print("Filas con region cargada:")
spark.sql(
    "SELECT * FROM demo.sales.orders WHERE region IS NOT NULL ORDER BY order_id"
).show(truncate=False)

print("Snapshots finales:")
spark.sql(
    "SELECT snapshot_id, operation, committed_at FROM demo.sales.orders.snapshots"
).show(truncate=False)

## Cierre

En esta sesión se vio:

| Capacidad | Qué acabamos de hacer |
|---|---|
| Object storage | Datos en MinIO (S3) |
| Catálogo | Metadatos Iceberg vía REST + Postgres |
| Time travel | Lectura de un snapshot anterior |
| Schema evolution | `ADD COLUMN` sin reescritura full |
| ACID | `UPDATE` versionado |

Opcional: revisar el bucket `warehouse` en la consola MinIO (`:9001`).